# Study correlation length-scale of atmospheric seeing

Some resources

- Piérre-François wrote some code inside this folder:  
  `/sdf/data/rubin/user/leget/lsst_dev/tickets/PFMeters/20251114`.

Associated tickets:
* [RSO-36](https://rubinobs.atlassian.net/browse/RSO-36)

In [ ]:
import glob
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import os
import pandas as pd
import pickle

from astropy.table import Table
from scipy.ndimage import binary_closing, binary_opening
from scipy.stats import binned_statistic_2d
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm  # just a simple progress bar

In [ ]:
# Number of spacial binning we want per detector
BINNING_PER_DETECTOR = 5

# Maximum number of detectors in a single row/column
N_DETECTORS = 15

# Colormap used through this notebook
CMAP = mpl.cm.Spectral.copy()
CMAP.set_bad("black")

## Reduced data

We already have some processed data living in `/sdf/data/rubin/user/leget/lsst_dev/tickets/PFMeters/visitPkls`.  
Instead of running the analysis from zero, let me try to play with these pickle files and see if I can extract anything useful.

In [ ]:
def convert_dict_with_tables_in_dataframe(my_dict : dict) -> pd.DataFrame:
    """
    Each item in the dictionary returned by the pickle file is an AstroPy Table,
    which can be easily converted into a Pandas DataFrame using a simple method.
    """
    df = pd.DataFrame()
    
    for key, val in my_dict.items():
 
        if val is None:
            continue
           
        sub_df = val.to_pandas()
        df = pd.concat([df, sub_df], ignore_index=True)

    return df

In [ ]:
def calculate_marker_size() -> float:
    """
    Calculates the marker size so it scales 
    with the number of bins per detector.
    """
    # Reference binning where s=15 works well
    BINNING_REF = 5  
    MARKER_SIZE_REF = 15

    # Calculate marker size that scales with binning
    marker_size = MARKER_SIZE_REF * (BINNING_REF / BINNING_PER_DETECTOR)**2

    return marker_size


In [ ]:
def calculate_color_normalization(data : np.array) -> mpl.colors.BoundaryNorm:
    """
    Finds the normalization to be applied into images and scatter plots.
    """
    vmin = np.nanmedian(data) - 2 * np.nanstd(data)
    vmax = np.nanmedian(data) + 2 * np.nanstd(data)

    bounds = np.round(np.linspace(vmin, vmax, 9), 1)
    norm = mpl.colors.BoundaryNorm(bounds, CMAP.N, extend='both')

    return norm

In [ ]:
def fill_detector_gaps_morphological(stats : np.array, structure_size:int=3, pad_width:int=10, verbose:bool=False):
    """
    Fill NaN gaps inside detector area using morphological operations.
    Uses padding to handle borders correctly.
    
    Parameters
    ----------
    stats : np.ndarray
        2D array with binned statistics (may contain NaN)
    structure_size : int
        Size of structuring element for morphological operations.
    pad_width : int
        Number of pixels to pad around the border (should be >= structure_size)
    verbose : bool
        Do you want to print messages?
    
    Returns
    -------
    np.ndarray
        Filled array
    """
    # Create binary mask: True where we have valid data
    valid_mask = ~np.isnan(stats)
    
    # Pad the valid mask with edge values (replicates border pixels)
    # This prevents edge effects during morphological operations
    padded_mask = np.pad(valid_mask, pad_width=pad_width, mode='edge')
    
    # Apply binary closing to fill small holes inside the detector area
    structure = np.ones((structure_size, structure_size))
    closed_mask_padded = binary_closing(padded_mask, structure=structure)
    
    # Optional: Apply opening to remove small noise outside
    # closed_mask_padded = binary_opening(closed_mask_padded, structure=structure)
    
    # Remove padding to get back to original size
    closed_mask = closed_mask_padded[pad_width:-pad_width, pad_width:-pad_width]
    
    # Identify gaps: pixels that are NaN but inside the closed detector area
    gaps_to_fill = closed_mask & np.isnan(stats)
    
    # Calculate global average from valid data
    global_avg = np.nanmean(stats[valid_mask])
    
    # Fill the gaps
    filled_stats = stats.copy()
    filled_stats[gaps_to_fill] = global_avg

    if verbose:
        print(f"Global average: {global_avg:.4f}")
        print(f"Filled {np.sum(gaps_to_fill)} gaps inside detector area")
        print(f"Kept {np.sum(~closed_mask & np.isnan(stats))} NaN values outside detector area")
    
    return filled_stats

### Single-Visit Analysis

Before doing anything, let me see if I can read the data and manipulate it a bit. 

In [ ]:
# Folder containing data processed by Pierre-François
PICKLE_FOLDER : str = "/sdf/data/rubin/user/leget/lsst_dev/tickets/PFMeters/visitPkls"

# Read all the pickle files inside that folder
list_of_filenames : list = [f for f in glob.glob(os.path.join(PICKLE_FOLDER, "*.pkl"))]

# Let's pay a bit with the first file. I will modify this code later.
filename : str = list_of_filenames[1]
print(f"Let's start our analysis with the file: {filename}")

# Let's open this pickle file
print(" Loading data...")
pickle_data : dict = pickle.load(open(filename, "rb"))
print(" Done!")

# I find it easier to work with Pandas DataFrames.
print(" Convert dict[table] into a dataframe.")
df : pd.DataFrame = convert_dict_with_tables_in_dataframe(pickle_data)
print(" Done!")

In [ ]:
# Grab filename from the original input
root_name, _ = os.path.splitext(os.path.split(filename)[-1])

# Let's pickup a column (we have many) to display
col = "T_src"

# Let's now calculate the minimum and maximum value based on the data itself
norm = calculate_color_normalization(df[col])

# Now let's prepare our figure to plot
fig, ax = plt.subplots(num=root_name, figsize=(12, 7), dpi=96)

sct = ax.scatter(df["xFoV"], df["yFoV"], c=df[col], s=1, cmap=CMAP, norm=norm)

cbar = ax.figure.colorbar(sct, ax=ax)
cbar.ax.set_ylabel(col, rotation=90)

# Add some cosmetics
ax.set_aspect('equal', adjustable='box') 
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_title(root_name)

fig.tight_layout()
fig.savefig(f"plots/{root_name}.png")

plt.show()

<br><br>  
Our data is sparse. We can fill the gaps using `binned_statistic_2d`. 

In [ ]:
# Get the total spacial binning (approximately)
spacial_binning = N_DETECTORS * BINNING_PER_DETECTOR

# Use `binned_statistic_2d` to fill up gaps with missing data
stats, x_edges, y_edges, bin_number = binned_statistic_2d(x=df["xFoV"], y=df["yFoV"], values=df[col], bins=spacial_binning)

# We need to rebuild a grid using the bin edges returned from the function above
x_center = 0.5 * (x_edges[1:] + x_edges[:-1])
y_center = 0.5 * (y_edges[1:] + y_edges[:-1])
Y, X = np.meshgrid(y_center, x_center)

# And now we can plot
fig, ax = plt.subplots(num=f"binned - {root_name}", figsize=(15, 7), dpi=96)

# Show the data
norm = calculate_color_normalization(stats)
marker_size = calculate_marker_size()
sct = ax.scatter(X, Y, c=stats, s=marker_size, cmap=CMAP, norm=norm)

cbar = ax.figure.colorbar(sct, ax=ax)
cbar.ax.set_ylabel(col, rotation=90)

# Add some cosmetics
ax.set_aspect('equal', adjustable='box') 
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_title(f"{root_name} Spacial Binned - n_bins per detector = {BINNING_PER_DETECTOR}")

fig.tight_layout()
fig.savefig(f"plots/stats_{root_name}.png")

plt.show()

<br><br>
Let's now normalize our data so it has an average of 0 mm and standard deviation of 1 mm.  
This will make it easier for the PCA to identify different components.  
Otherwise, the first PCA component will most likely be the average. 

In [ ]:
# Standard scaler that bring the average of our dataset to zero and
# normalizes it so the standard deviation is 1
scaler = StandardScaler()

# Fit and transform the data
scaled_stats = scaler.fit_transform(stats)

# And now we can plot
fig, ax = plt.subplots(num=f"binned and normalized - {root_name}", figsize=(15, 7), dpi=96)

# Show the data
norm = calculate_color_normalization(scaled_stats)
marker_size = calculate_marker_size()
sct = ax.scatter(X, Y, c=scaled_stats, s=marker_size, cmap=CMAP, norm=norm)

cbar = ax.figure.colorbar(sct, ax=ax)
cbar.ax.set_ylabel(col, rotation=90)

# Add some cosmetics
ax.set_aspect('equal', adjustable='box') 
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_title(f"{root_name} Spacial Binned - n_bins per detector = {BINNING_PER_DETECTOR}")

fig.tight_layout()
fig.savefig(f"plots/stats_{root_name}.png")

plt.show()

<br><br>
Let's plot both histograms for comparison: before and after scaling.  
The spacial structure changes a lot. 

In [ ]:
fig, (ax1, ax2) = plt.subplots(num=f"histogram - {root_name}", figsize=(15, 3), dpi=96, ncols=2)

ax1.hist(stats.ravel(), bins=50, fc="C0", ec="white")
ax1.set_title("Binned data")
ax1.set_xlabel("T_src [px]")
ax1.grid(":", alpha=0.2)

ax2.hist(scaled_stats.ravel(), bins=50, fc="C1", ec="white")
ax2.set_title("Binned and scaled data")
ax2.set_xlabel("Normalized T_src [px]")
ax2.grid(":", alpha=0.2)

fig.suptitle(f"Histograms {root_name}")
plt.show()


<br><br>
Now we need to deal with the missing data.  
There are several strategies here.  
One of them can be using morphological operations.  
Remember that these morphological operations require padding to work properly.  
Before implementing anything, let's see how it works.

In [ ]:
# Visualize what's being filled
fig, axes = plt.subplots(2, 3, figsize=(12, 7))

# Parameters
pad_width = 25
structure_size = 10
structure = np.ones((structure_size, structure_size))

# Original valid mask
valid_mask = ~np.isnan(stats)

# Padded mask
padded_mask = np.pad(valid_mask, pad_width=pad_width, mode='edge')

# Closed mask (padded)
closed_mask_padded = binary_closing(padded_mask, structure=structure)

# Remove padding
closed_mask = closed_mask_padded[pad_width:-pad_width, pad_width:-pad_width]

# Identify gaps
gaps = closed_mask & np.isnan(stats)

# Plotting
axes[0, 0].imshow(valid_mask, cmap='gray')
axes[0, 0].set_title('Original Valid Data')
axes[0, 0].set_aspect('equal')

axes[0, 1].imshow(padded_mask, cmap='gray')
axes[0, 1].set_title(f'After Padding (pad_width={pad_width})')
axes[0, 1].set_aspect('equal')

axes[0, 2].imshow(closed_mask_padded, cmap='gray')
axes[0, 2].set_title(f'After Closing (padded, size={structure_size})')
axes[0, 2].set_aspect('equal')

axes[1, 0].imshow(closed_mask, cmap='gray')
axes[1, 0].set_title('After Removing Padding')
axes[1, 0].set_aspect('equal')

axes[1, 1].imshow(gaps, cmap='Reds')
axes[1, 1].set_title('Gaps to Fill (red)')
axes[1, 1].set_aspect('equal')

# Show original data with gaps highlighted
axes[1, 2].imshow(np.isnan(stats), cmap='Reds', alpha=0.5)
axes[1, 2].imshow(valid_mask, cmap='gray', alpha=0.5)
axes[1, 2].set_title('Original NaN locations (red)')
axes[1, 2].set_aspect('equal')

plt.tight_layout()
plt.show()

<br><br>
Cool! I believe we can move on and implement it.  
The implementation is in a function up above called `fill_detector_gaps_morphological`.  

In [ ]:
# Option 1 - Fill up the gaps with the average
stats_filled = scaled_stats.copy()
stats_filled[np.isnan(stats_filled)] = np.nanmean(stats_filled)

# Option 2 - Fill up the gaps using morphological operations
# stats_filled = fill_detector_gaps_morphological(scaled_stats, structure_size=7, pad_width=25, verbose=True)

# And now we can plot
fig, ax = plt.subplots(num=f"binned and imputed - {root_name}", figsize=(15, 7), dpi=96)

# Show the data
marker_size = calculate_marker_size()
norm = calculate_color_normalization(stats_filled)
sct = ax.scatter(X, Y, c=stats_filled, s=marker_size, cmap=CMAP, norm=norm)

cbar = ax.figure.colorbar(sct, ax=ax)
cbar.ax.set_ylabel(col, rotation=90)

# Add some cosmetics
ax.set_aspect('equal', adjustable='box') 
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_title(f"{root_name} Binned and Imputed Data\nn_bins per detector = {BINNING_PER_DETECTOR}")

fig.tight_layout()
fig.savefig(f"plots/stats_{root_name}.png")

plt.show()

### Multi-Visit Analysis

Right. Now that I have a better understanding of the data itself, let's do some patch processing.  
I will read each file, convert the dictionary containing tables into a pandas dataframe,  
extract the `T_src` column (what does it mean?),  
calculate the binned 2d statistics to augment the data,  
reshape the array so it is a single dimension array,
append the result to another array.  

The final result must be a 2D array where each row corresponds to the data associated to a single visit.  
The columns are the binned `T_src` squeezed in a single dimention.

In [ ]:
# Folder containing data processed by Pierre-François
PICKLE_FOLDER : str = "/sdf/data/rubin/user/leget/lsst_dev/tickets/PFMeters/visitPkls"

# Read all the pickle files inside that folder
list_of_filenames : list = [f for f in glob.glob(os.path.join(PICKLE_FOLDER, "*.pkl"))]

# Let's work with a single column for now
col = "T_src"

# Let's use n_bins in both X and Y
n_bins = N_DETECTORS * BINNING_PER_DETECTOR

# Let's ensure that our scaler is initialized
scaler = StandardScaler()

# Stacked data - contains data from all the visits
stacked_data = []

# Loop over each file
for pickle_fname in tqdm(list_of_filenames):
  
    _pickle_data : dict = pickle.load(open(pickle_fname, "rb"))
    _df : pd.DataFrame = convert_dict_with_tables_in_dataframe(_pickle_data)

    if _df.index.size == 0:
        continue
    
    _x : pd.Series = _df["xFoV"]
    _y : pd.Series = _df["yFoV"]
    _data : pd.Series = _df[col]

    # 2d binning
    _stats, _, _, _ = binned_statistic_2d(x=_x, y=_y, values=_data, bins=n_bins)

    # Remove average and scale to have std = 1
    _scaled_stats = scaler.fit_transform(_stats)

    # Option 1 - Fill all the missing data with the average
    _imputed_stats = _scaled_stats.copy()
    _imputed_stats[np.isnan(_imputed_stats)] = np.nanmean(_imputed_stats)
    
    # Option 2 (Not working yet) - Use morphological operations to fill only the gaps where data is missing
    # _imputed_stats = fill_detector_gaps_morphological(_scaled_stats, structure_size=7, pad_width=25)

    # "Unroll" the data and make it 1D.
    _imputed_stats = _imputed_stats.ravel() 

    # Attach data
    stacked_data.append(_imputed_stats)


# Convert everything back into a 2D array. 
# Each row corresponds to the data of a single visit. 
# Each column corresponds to the binned data into a particular bin.
stacked_data = np.array(stacked_data)

<br><br>
Our data has lots of NaNs and gaps.  
Use the code below to fill up the gaps. 

In [ ]:
# Option 1 - All the arrays are filled up with valid data. No need for further processing
pass

# Option 2 - Not implemented - We have missing data outside the detector area. 
# To Do

# Old code - I used the steps below to clear missing data in a older version of this notebook. 

# # Step 1: Find and remove all-NaN columns
all_nan_cols = np.all(np.isnan(stacked_data), axis=0)
nan_col_indices = np.where(all_nan_cols)[0]
np.save('nan_columns.npy', nan_col_indices)  # Save for later

# # Step 2: Keep only columns with at least some data
data_clean = stacked_data[:, ~all_nan_cols]

# # Step 3: Now impute the remaining NaNs
# imputer = SimpleImputer(strategy='mean')
# data_imputed = imputer.fit_transform(data_clean)

# # data_imputed = data_clean.copy()
# # data_imputed[np.isnan(data_imputed)] = np.nanmean(data_imputed)

<br><br>
Now that we have our data in a specific format, let's try to apply PCA to it.

In [ ]:
# Step 4: Apply PCA
pca = PCA(n_components=50)
pca.fit(stacked_data)

<br><br>
Review the PCA with some metrics and plots.

In [ ]:
# 1. Explained variance per component
print(pca.explained_variance_ratio_)

# 2. Cumulative explained variance
cumsum = np.cumsum(pca.explained_variance_ratio_)
print(f"50 components explain {cumsum[-1]:.1%} of variance")

# 3. Scree plot (see elbow)
plt.semilogy(pca.explained_variance_ratio_)
plt.xlabel('Component')
plt.ylabel('Explained Variance Ratio')
plt.savefig("plots/explained_variance_ratio.png")
plt.show()


# 4. Cumulative variance plot
plt.plot(cumsum)
plt.axhline(y=0.95, color='r', linestyle='--')  # 95% threshold
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.savefig("plots/cumulative_explained_variance.png")
plt.show()


# 5. Reconstruction error (if needed)
reconstructed = pca.inverse_transform(pca.transform(stacked_data))
mse = np.mean((stacked_data - reconstructed)**2)

In [ ]:
# Reshape components back to your grid shape
n_bins_x, n_bins_y = n_bins, n_bins  # or whatever your grid size is

fig, axes = plt.subplots(10, 2, figsize=(15, 30), num="pca_analysis")

for i, ax in enumerate(axes.flat):
    
    # Reconstruct the component with NaN columns added back
    component_full = np.zeros(stacked_data.shape[1])
    component_full[~all_nan_cols] = pca.components_[i]
    component_full[all_nan_cols] = np.nan

    norm = calculate_color_normalization(component_full)

    _vmin = np.nanmedian(component_full) - 2 * np.nanstd(component_full)
    _vmax = np.nanmedian(component_full) + 2 * np.nanstd(component_full)

    _bounds = np.round(np.linspace(_vmin, _vmax, 9), 1)
    _norm = mpl.colors.BoundaryNorm(_bounds, CMAP.N, extend='both')
    
    # Reshape to spatial grid
    pc_map = component_full.reshape(n_bins_y, n_bins_x)
    pc_map = np.ma.masked_invalid(pc_map)
    pc_map = pc_map[::-1, ::-1] # Get the orientation right
    
    im = ax.imshow(pc_map, cmap=CMAP, vmin=_vmin, vmax=_vmax, origin="lower")
    
    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.ax.set_ylabel(col, rotation=90)

    ax.set_title(f'PC{i+1} ({pca.explained_variance_ratio_[i]:.1%})')  
    
    
fig.tight_layout()

root_fname, _ = os.path.splitext(os.path.split(pickle_fname)[-1])
fig.savefig(f"plots/pca_{root_fname[:8]}.png")